In [1]:
#Install necessary packages
%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity

Note: you may need to restart the kernel to use updated packages.


In [13]:
#Setting up environment variables

import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, AzureAISearchAgentTool, AzureAISearchToolResource, AzureAISearchQueryType,AISearchIndexResource

load_dotenv()

foundry_project_endpoint =os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")
ai_search_connection_name=os.getenv("AI_SEARCH_CONNECTION_NAME")
ai_search_index_name=os.getenv("AI_SEARCH_INDEX_NAME")


In [18]:
project_client= AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential()
)

In [19]:
openai_client= project_client.get_openai_client()

Fetching the AI Search Connection ID

In [20]:
#Getting the MCP Server connection ID
connection_id=""

for connection in project_client.connections.list():
    if connection.name == ai_search_connection_name:
        connection_id = connection.id
        break

print(f"MCP Server Connection ID: {connection_id}")

MCP Server Connection ID: /subscriptions/66a53095-4447-4d75-8e00-78373b92da78/resourceGroups/foundry_demo_grp/providers/Microsoft.CognitiveServices/accounts/deepanshu8884-1528-resource/projects/deepanshu8884-2709/connections/ragaisearchdemo16n0me0


In [27]:
#Creating the RAG Agent

agent = project_client.agents.create_version(
    agent_name="RAG-Agent",
    definition=PromptAgentDefinition(
        model=model_deployment_name,
        instructions= "you are a helpful assistant that uses AI search to answer queries in a RAG setup and deny to each questions which are not part of ai search knowledge source.",
        tools=[
            AzureAISearchAgentTool(
                azure_ai_search=AzureAISearchToolResource(
                    indexes=[
                        AISearchIndexResource(
                            project_connection_id=connection_id,
                            index_name=ai_search_index_name,
                            query_type=AzureAISearchQueryType.VECTOR_SEMANTIC_HYBRID,
                            top_k=3
                        )
                    ]
                )
            )
        ]
    )    
)

In [28]:
conversation= openai_client.conversations.create()


In [ ]:
#callin the agent and streaming the response
user_input="what are the hotels offered by margie's travel in las vegas? state the Sources also please."

In [37]:
response = openai_client.responses.create(
    tool_choice="required",
    conversation=conversation.id,
    input=user_input,
    extra_body={"agent":{"name":agent.name, "type":"agent_reference"}}
)

print(f"Agent responses: {response.output_text}")

Agent responses: The search in the knowledge source did not return information about the "best snack in the world." However, the "best snack" can be subjective and varies depending on personal taste, culture, and region. Popular snacks globally include items like chips, pretzels, nuts, fruit snacks, and traditional snacks like samosas, dumplings, or tapas. If you want recommendations for a specific type of snack or from a particular cuisine, please let me know!
